In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
df_train = pd.read_csv("/kaggle/input/compressed-cic-2018/df_train.csv")
df_test = pd.read_csv("/kaggle/input/compressed-cic-2018/df_test.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nTrain label distribution:\n{df_train['label'].value_counts()}")
print(f"\nTest label distribution:\n{df_test['label'].value_counts()}")

In [ ]:
# Prepare features and labels
feature_cols = ['latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'recon_loss', 'kld_loss']

X_train_full = df_train[feature_cols].values
y_train_full = df_train['label'].values

X_test = df_test[feature_cols].values
y_test = df_test['label'].values

# Compute inverse class weights
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train_full)

print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")
print(f"\nClass distribution in training:")
unique, counts = np.unique(y_train_full, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f"  Class {c}: {cnt} samples, weight: {len(y_train_full) / (len(unique) * cnt):.4f}")

# Optuna Hyperparameter Tuning

In [ ]:
def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': len(np.unique(y_train_full)),
        'metric': 'multi_logloss',
        'device': 'gpu',
        'verbosity': -1,
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 100.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 100.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 2.0),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_train_fold = X_train_full[train_idx]
        y_train_fold = y_train_full[train_idx]
        X_val_fold = X_train_full[val_idx]
        y_val_fold = y_train_full[val_idx]
        sw_train_fold = sample_weights_full[train_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_fold, y_train_fold,
            sample_weight=sw_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
        )
        
        y_pred = model.predict(X_val_fold)
        f1 = f1_score(y_val_fold, y_pred, average='weighted')
        f1_scores.append(f1)
    
    return np.mean(f1_scores)

# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True, catch=(Exception,))

print(f"\nBest trial F1: {study.best_trial.value:.4f}")
print(f"Best params: {study.best_trial.params}")

# Train Final Model with Best Params

In [ ]:
best_params = study.best_trial.params
best_params.update({
    'objective': 'multiclass',
    'num_class': len(np.unique(y_train_full)),
    'metric': 'multi_logloss',
    'device': 'gpu',
    'verbosity': -1,
    'random_state': 42,
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train_full, y_train_full, sample_weight=sample_weights_full)

print("Final model trained on full training data with class balancing!")

# Final Evaluation on Test Set

In [ ]:
# Evaluate on test set
y_pred_test = final_model.predict(X_test)

# Compute sample weights for test set (multiclass)
sample_weights_test = compute_sample_weight(class_weight='balanced', y=y_test)

# Create binary labels for test: 0 = benign, 1 = attack (any non-0)
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred_test != 0).astype(int)

# Compute binary class weights (inverse frequency for 0 vs 1)
sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)

# Custom weighted binary accuracy
def weighted_binary_accuracy(y_true_bin, y_pred_bin, sample_weights):
    correct_mask = y_true_bin == y_pred_bin
    
    benign_mask = y_true_bin == 0
    attack_mask = y_true_bin == 1
    
    benign_correct_weighted = np.sum(sample_weights[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights[benign_mask])
    
    attack_correct_weighted = np.sum(sample_weights[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights[attack_mask])
    
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights)
    
    print(f"\nWeighted Binary Evaluation (Attack vs Benign):")
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    
    return total_correct_weighted / total_weighted

print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

# Binary class distribution
n_benign = np.sum(y_test_binary == 0)
n_attack = np.sum(y_test_binary == 1)
print(f"\nBinary test distribution: Benign={n_benign}, Attack={n_attack}")
print(f"Binary weights: Benign={len(y_test_binary)/(2*n_benign):.4f}, Attack={len(y_test_binary)/(2*n_attack):.4f}")

print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred_test, sample_weight=sample_weights_test):.4f}")
print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
weighted_bin_acc = weighted_binary_accuracy(y_test_binary, y_pred_binary, sample_weights_binary)
print(f"Accuracy (binary weighted): {weighted_bin_acc:.4f}")

print(f"\nF1 (weighted): {f1_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Precision (weighted): {precision_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred_test, average='weighted'):.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Multiclass)")
print("=" * 60)
print(classification_report(y_test, y_pred_test))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
print("=" * 60)
print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

In [ ]:
# Save the model
final_model.booster_.save_model("/kaggle/working/lightgbm_model.txt")
print("Model saved to /kaggle/working/lightgbm_model.txt")